# Incubator State Estimation with a Particle Filter

Next we contruct the kalman filter, using the system sympy library to transform the system symbolic equations to a discrete time system.
We adjust the equations from [ModellingIncubatorDynamics](../3-Physics-Modelling/2-ModellingIncubatorDynamics.ipynb).

In [ ]:
import sympy as sp
import numpy as np

import sympy as sp
import numpy as np

# Parameters
C_air = sp.Symbol("C_air", real=True)  # Specific heat capacity
G_box = sp.Symbol("G_box", real=True)  # Specific heat capacity
C_heater = sp.Symbol("C_heater", real=True)  # Specific heat capacity
G_heater = sp.Symbol("G_heater", real=True)  # Specific heat capacity

# Constants
V_heater = sp.Symbol("V_heater", real=True)
i_heater = sp.Symbol("i_heater", real=True)

# Inputs
in_room_temperature = sp.Symbol("T_room", real=True)
on_heater = sp.Symbol("on_heater", real=True)

# States
T = sp.Symbol("T", real=True)
T_heater = sp.Symbol("T_heater", real=True)

power_in = on_heater * V_heater * i_heater

power_transfer_heat = G_heater * (T_heater - T)

total_power_heater = power_in - power_transfer_heat

power_out_box = G_box * (T - in_room_temperature)

total_power_box = power_transfer_heat - power_out_box

der_T_heater = (1.0 / C_heater) * (total_power_heater)
der_T = (1.0 / C_air) * (total_power_box)

# Vectorize der_T and der_T_heater so they can be applied to the particles
der_T_heater_function = sp.lambdify((T_heater, T, C_air, G_box, C_heater, G_heater, V_heater, i_heater, in_room_temperature, on_heater), der_T_heater)
der_T_function = sp.lambdify((T_heater, T, C_air, G_box, C_heater, G_heater, V_heater, i_heater, in_room_temperature, on_heater), der_T)

In [ ]:
# System parameters
V_heater_num = HEATER_VOLTAGE = 12.0
i_heater_num = HEATER_CURRENT = 10.45
C_air_num = 68.21
G_box_num = 0.74 
C_heater_num = 243.46
G_heater_num = 0.87

In [ ]:
from data_handling import load_data
import pandas
import math

# Load the data
time_unit = 'ns'
data, _ = load_data("./lid_opening_experiment_jan_2021/lid_opening_experiment_jan_2021.csv",
                    HEATER_VOLTAGE, HEATER_CURRENT,
                    desired_timeframe=(- math.inf, math.inf),
                    time_unit=time_unit,
                    normalize_time=False,
                    convert_to_seconds=True)
events = pandas.read_csv("./lid_opening_experiment_jan_2021/events.csv")
events["timestamp_ns"] = pandas.to_datetime(events["time"], unit=time_unit)

data

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1)

ax1.plot(data["time"], data["t1"], label="t1")
ax1.plot(data["time"], data["t2"], label="t2")
ax1.plot(data["time"], data["t3"], label="t3")
ax1.plot(data["time"], data["average_temperature"], label="average_temperature")
ax1.legend()

ax2.plot(data["time"], data["heater_on"], label="heater_on")
ax2.plot(data["time"], data["fan_on"], label="fan_on")
ax2.legend()

ax3.plot(data["time"], data["execution_interval"], label="execution_interval")
ax3.plot(data["time"], data["elapsed"], label="elapsed")
ax3.legend()

ax4.plot(data["time"], data["power_in"], label="power_in")
ax4.legend()

In [ ]:
events

In [ ]:
# Inputs to _plant
measurements_heater = np.array([1.0 if b else 0.0 for b in data["heater_on"]])
measurements_Troom = data["t1"].to_numpy()

# System state measurements (partial)
measurements_T = data["average_temperature"].to_numpy()

# Initial particles. Each particle represents a possible system state.
N = 100
particles = np.zeros((N, 2))
particles[:, 0] = measurements_T[0]  # Initial heater temperature
particles[:, 1] = measurements_T[0]  # Initial box temperature

weights = np.ones(N) / N  # Uniform weights

observation_noise_std = 0.1 # Comes from temperature sensor datasheet
process_noise_std = 0.3 # Can be tuned to, e.g., better detect that lid has been opened.

# Show how vectorized functions work:
der_T_heater_function(particles[:, 0], particles[:, 1], C_air_num, G_box_num, C_heater_num, G_heater_num, V_heater_num, i_heater_num, measurements_Troom[0], measurements_heater[0])

In [ ]:
# Main simulation loop

# results

results = np.zeros((len(data['time']), 2))
uncertainty_estimate = np.zeros((len(data['time']), 2))

for step in range(len(data['time'])):
    results[step, 0] = np.mean(particles[:, 0]) # heater temperature
    results[step, 1] = np.mean(particles[:, 1]) # box temperature
    uncertainty_estimate[step] = np.std(particles[:, 0])

    dt = data['time'][step] - data['time'][step - 1] if step > 0 else 3.0
    particles[:, 0] += der_T_heater_function(particles[:, 0], particles[:, 1], C_air_num, G_box_num, C_heater_num, G_heater_num, V_heater_num, i_heater_num, measurements_Troom[step], measurements_heater[step])*dt + np.random.normal(0, process_noise_std, N)

    # note that we're using the updated heater temperature here. This is a slightly different numerical method but simple to write and doesn't affect the results much.
    particles[:, 1] += der_T_function(particles[:, 0], particles[:, 1], C_air_num, G_box_num, C_heater_num, G_heater_num, V_heater_num, i_heater_num, measurements_Troom[step], measurements_heater[step])*dt + np.random.normal(0, process_noise_std, N)

    observation = measurements_T[step]

    # Update weights based on observation likelihood
    weights = np.exp(-0.5 * ((particles[:, 1] - observation) / observation_noise_std) ** 2)
    weights += 1e-300  # Avoid zero weights, just for numerical stability
    weights /= np.sum(weights)  # Normalize: turn the weights into a probability distribution

    # Resample particles
    indices = np.random.choice(N, N, p=weights)
    particles = particles[indices, :]
    weights = np.ones(N) / N  # Reset weights


results

In [ ]:
uncertainty_estimate

In [ ]:
from data_handling import plotly_incubator_data

fig = plotly_incubator_data(data,
                            compare_to={
                                "PF": {
                                    "timestamp_ns": data["timestamp_ns"],
                                    "T": results[:, 1],
                                    "T_std_dev": uncertainty_estimate[:, 1]
                                },
                            },
                            heater_T_data={
                                "PF": {
                                    "timestamp_ns": data["timestamp_ns"],
                                    "T_heater": results[:, 0],
                                    "T_heater_std_dev": uncertainty_estimate[:, 0]
                                },
                            },
                            events=events,
                            overlay_heater=True,
                            show_hr_time=True)

# Save plotly interactive plot
import plotly.io as pio
pio.write_html(fig, file="incubator_PF.html")

fig